# Colab Training Runbook

GPU launcher for the official thesis curriculum experiments. `configs/{model}.yaml` trains on `data/output/train.h5` and validates on `data/output/val.h5`; evaluation runs against `data/output/test.h5`.

## Before Running

1. Select a GPU runtime: Runtime > Change runtime type > GPU.
2. Put train.h5, val.h5, and test.h5 under MyDrive/masters-thesis/data/output/.
3. Paste a GitHub token in Step 1 if the repo is private.
4. Keep GIT_REF on the branch that contains the current evaluator schema.

## Drive layout

```text
runs/<model>/<run_id>/
```

Each completed run writes best.pt, latest.pt, metrics.csv, evaluation/summary.csv, and evaluation/metrics.json.

In [ ]:
# @title 1. Setup: Drive, experiment, and repo
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")


def _shell(*cmd: str) -> None:
    """Run a command, streaming stdout/stderr live to the cell, raising on failure.

    Why Popen + manual line iteration instead of subprocess.run: the OS-level
    stdout file descriptor inherited from the kernel does not stream to the
    Colab cell in real time. Piping through Python and printing each line
    routes output via Jupyter's wrapped sys.stdout, which is the cell.
    """
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if proc.stdout is not None:
        for line in proc.stdout:
            print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)


GITHUB_TOKEN = ""  # @param {type:"string"}
REPO = "AlexNeagu123/msc-thesis-gnn-nbody"
GIT_REF = "main"  # @param {type:"string"}
DRIVE = Path("/content/drive/MyDrive/masters-thesis")

MODEL = "egnn"  # @param ["egnn", "hgnn"]
N_TRAIN = "1000"  # @param ["1000"]
RUN_TRAINING = True  # @param {type:"boolean"}
RUN_EVALUATION = True  # @param {type:"boolean"}
SKIP_COMPLETED = False  # @param {type:"boolean"}

N_TRAIN = int(N_TRAIN)

assert MODEL in {"egnn", "hgnn"}, MODEL
assert N_TRAIN == 1000, N_TRAIN

DATA_PATH_PREFIX = "data/output"
if not GITHUB_TOKEN:
    raise ValueError("Paste a GitHub token before cloning the repo.")

repo_dir = Path("/content/repo")
clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"

if (repo_dir / ".git").exists():
    _shell("git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", GIT_REF)
    _shell("git", "-C", str(repo_dir), "checkout", "FETCH_HEAD")
elif repo_dir.exists():
    raise FileExistsError(f"{repo_dir} exists but is not a git checkout")
else:
    _shell("git", "clone", "--depth", "1", "--branch", GIT_REF, clone_url, str(repo_dir))

project_dir = repo_dir / "impl" if (repo_dir / "impl" / "requirements.txt").exists() else repo_dir
if not (project_dir / "requirements.txt").exists():
    raise FileNotFoundError(
        f"Could not find requirements.txt under {repo_dir} or {repo_dir / 'impl'}"
    )

get_ipython().run_line_magic("cd", str(project_dir))
print(f"Project directory: {project_dir}")
print(f"Model: {MODEL}")
print(f"Dataset: {DATA_PATH_PREFIX}")

In [ ]:
# @title 2. Install dependencies and verify GPU
_shell("pip", "install", "-q", "-r", "requirements.txt")

import torch  # noqa: E402

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Select a GPU runtime before continuing.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# @title 3. Copy dataset from Drive and verify shapes
import shutil

import h5py

data_dir = Path(DATA_PATH_PREFIX)
data_dir.mkdir(parents=True, exist_ok=True)

for name in ["train.h5", "val.h5", "test.h5"]:
    src = DRIVE / "data" / "output" / name
    dst = data_dir / name
    if not src.exists():
        raise FileNotFoundError(f"Missing Drive data file: {src}")
    shutil.copy2(src, dst)

for name in ["train.h5", "val.h5", "test.h5"]:
    with h5py.File(data_dir / name, "r") as f:
        shape = f["trajectories"].shape
        print(f"{name}: {shape}")
        if name == "train.h5" and shape[0] < N_TRAIN:
            raise ValueError(f"train.h5 has {shape[0]} trajectories, but N_TRAIN={N_TRAIN}")

In [ ]:
# @title 4. Write Colab config pointing at Drive paths
import yaml

RUN_ROOT = DRIVE / "runs" / MODEL
RUN_ROOT.mkdir(parents=True, exist_ok=True)

src_cfg_path = Path(f"configs/{MODEL}.yaml")
cfg = yaml.safe_load(src_cfg_path.read_text())
cfg["data"]["train_path"] = f"{DATA_PATH_PREFIX}/train.h5"
cfg["data"]["val_path"] = f"{DATA_PATH_PREFIX}/val.h5"
cfg["training"]["device"] = "auto"
cfg["checkpointing"]["enabled"] = True
cfg["checkpointing"]["dir"] = str(RUN_ROOT)
cfg["logging"]["enabled"] = True
cfg["logging"]["dir"] = str(RUN_ROOT)

CONFIG_PATH = Path("configs") / f"_colab_{MODEL}_n{N_TRAIN}.yaml"
CONFIG_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(f"run_root: {RUN_ROOT}")
print(f"config:   {CONFIG_PATH}")

In [ ]:
# @title 5. Train and evaluate
def _train_cmd() -> list[str]:
    return [
        "python",
        "-u",
        "-m",
        "training.train",
        "--config",
        str(CONFIG_PATH),
        "--n-train",
        str(N_TRAIN),
    ]


def _eval_cmd(checkpoint: Path, eval_dir: Path) -> list[str]:
    return [
        "python",
        "-u",
        "-m",
        "evaluation.evaluate",
        "--config",
        str(CONFIG_PATH),
        "--checkpoint",
        str(checkpoint),
        "--test-path",
        f"{DATA_PATH_PREFIX}/test.h5",
        "--output-dir",
        str(eval_dir),
        "--device",
        "cuda",
    ]


existing = sorted(
    p for p in RUN_ROOT.iterdir() if p.is_dir() and (p / "evaluation" / "metrics.json").exists()
)
if SKIP_COMPLETED and existing:
    run_dir = existing[-1]
    print(f"skipping completed run: {run_dir}")
elif RUN_TRAINING:
    print(f"training {MODEL} N={N_TRAIN}; output={RUN_ROOT}", flush=True)
    _shell(*_train_cmd())
    run_ids = sorted(
        p.name for p in RUN_ROOT.iterdir() if p.is_dir() and (p / "latest.pt").exists()
    )
    if not run_ids:
        raise RuntimeError(f"No checkpoint runs found under {RUN_ROOT}.")
    run_dir = RUN_ROOT / run_ids[-1]
else:
    run_ids = sorted(
        p.name for p in RUN_ROOT.iterdir() if p.is_dir() and (p / "latest.pt").exists()
    )
    if not run_ids:
        raise RuntimeError(f"Training skipped and no checkpoint runs found under {RUN_ROOT}.")
    run_dir = RUN_ROOT / run_ids[-1]

BEST_CHECKPOINT = run_dir / "best.pt"
LATEST_CHECKPOINT = run_dir / "latest.pt"
EVAL_DIR = run_dir / "evaluation"
print(f"ready: run_dir={run_dir}")

if RUN_EVALUATION:
    print(f"evaluating checkpoint={BEST_CHECKPOINT}", flush=True)
    _shell(*_eval_cmd(BEST_CHECKPOINT, EVAL_DIR))
else:
    print("Evaluation skipped.")

In [ ]:
# @title 6. Print evaluation summary
summary_path = EVAL_DIR / "summary.csv"
print(f"Evaluation directory: {EVAL_DIR}")
if summary_path.exists():
    print(summary_path.read_text())

In [ ]:
# @title 7. Verify persistent Drive artifacts
print(f"model={MODEL}  run={run_dir}")
items = [
    BEST_CHECKPOINT,
    LATEST_CHECKPOINT,
    run_dir / "metrics.csv",
    run_dir / "diagnostics.log",
    EVAL_DIR / "summary.csv",
    EVAL_DIR / "metrics.json",
]
for path in items:
    status = "ok" if path.exists() else "missing"
    print(f"{status}: {path}")